In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from pandas import DataFrame
from datasets import load_dataset as hf_load_dataset
import json

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from collections import defaultdict


/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
linting_paths = {
    "php": "../../results/php_lint_conversation_redo.json",
    "csharp": "../../results/csharp_lint_conversation_redo.json",
    "python": "../../results/python_lint_conversation_redo.json",
    "c": "../../results/c_lint_conversation_redo.json",
    "javascript": "../../results/js_lint_conversation_redo.json",
    "java": "../../results/java_lint_conversation_redo.json",
}

In [3]:
other_cleaned = []
with open("tmp/other_cleaned.json", "r", encoding="utf-8") as f:
    for line in f.readlines():
        other_cleaned.append(json.loads(line))

In [4]:
tracker = {
    "php": 0,
    "csharp": 0,
    "python": 0,
    "c": 0,
    "javascript": 0,
    "java": 0,
}
for row in other_cleaned:
    language = row["language"].lower()
    if language == "c++":
        language = "c"
    elif language == "c#":
        language = "csharp"
    if language not in tracker.keys():
        continue
    tracker[language] += 1


print(tracker)

{'php': 1935, 'csharp': 13893, 'python': 57371, 'c': 21510, 'javascript': 26442, 'java': 19257}


In [5]:
linting_results = {}

for key, value in linting_paths.items():
    with open(value, "r", encoding="utf-8") as f:
        linting_results[key] = json.load(f)

In [6]:
for key, value in linting_results.items():
    hashes = set()
    for row in value:
        hashes.add(row["module"])
    print(key, len(hashes))

php 125
csharp 649
python 5281
c 1400
javascript 4123
java 1


In [7]:
messages = []

for key, value in linting_results.items():
    for row in value:
        convo_hash = row["module"].split("_")[0]
        messages.append(row["message"])

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2", backend="openvino")

Multiple OpenVINO files found in 'sentence-transformers/all-MiniLM-L6-v2': ['openvino/openvino_model.xml', 'openvino/openvino_model_qint8_quantized.xml'], defaulting to 'openvino/openvino_model.xml'. Please specify the desired file name via `model_kwargs={"file_name": "<file_name>"}`.


In [10]:
# Step 1: embed raw messages
embeddings = model.encode(messages, batch_size = 128)
np.save("../../tmp/embeddings.npy", embeddings)

In [11]:
embeddings[0]

array([ 2.46849880e-02,  4.39816490e-02, -3.14820521e-02, -6.85311779e-02,
        9.41391140e-02, -2.75903139e-02, -3.23137902e-02, -5.57325743e-02,
        7.75558650e-02, -4.13013343e-03,  4.45948131e-02,  2.82807611e-02,
       -5.99922903e-04, -4.41075116e-02,  9.90915345e-04,  1.53638609e-02,
       -9.76782814e-02, -7.15874732e-02,  2.88205296e-02,  2.09799726e-02,
       -2.53869351e-02,  3.44959386e-02,  1.52326524e-02,  5.81550449e-02,
        5.43988012e-02,  3.62654291e-02, -1.60212107e-02,  5.50581478e-02,
       -2.51655048e-03,  3.39220688e-02,  1.80487335e-02,  5.71943186e-02,
        1.11863492e-02, -1.36825221e-03,  4.39815111e-02,  4.86648083e-02,
        3.53345945e-02, -4.92047183e-02,  1.36351015e-03,  1.41614610e-02,
       -5.35821170e-02,  5.53096645e-03, -2.12305179e-03,  5.05858473e-02,
       -2.84665059e-02, -1.33839790e-02,  3.10898386e-02, -8.72898400e-02,
       -4.70301993e-02, -6.22191355e-02, -6.50401860e-02, -2.60790065e-02,
        3.12494561e-02,  

In [13]:
embeddings = np.load("../../tmp/embeddings.npy")

In [14]:
error_categories = []
with open("../../utils/error_categories.json", "r", encoding="utf-8") as f:
    error_categories = json.load(f)

In [15]:
len(error_categories)

20

In [16]:
error_category_embeddings = {}
for error_category in error_categories:
    
    description = error_category["description"]
    embedding = model.encode([description])
    error_category_embeddings[error_category["category"]] = embedding


In [ ]:
# Cosine similarity algorithm on the categories vs error messages

In [17]:
categories = list(error_category_embeddings.keys())
category_matrix = np.vstack([error_category_embeddings[c] for c in categories])

similarity = cosine_similarity(embeddings, category_matrix)

print(similarity.shape)

best_indices = np.argmax(similarity, axis=1)
best_categories = [categories[idx] for idx in best_indices]

results = []
for msg, cat_idx in zip(messages, best_indices):
    results.append({"message": msg, "predicted_category": categories[cat_idx]})

(24257, 20)


In [18]:
import random

# Step 1: assign each message to its best category, carrying the language key
category_to_lang_msgs = defaultdict(lambda: defaultdict(list))

# Flatten but keep language info
all_rows = []
for lang, rows in linting_results.items():
    for row, cat in zip(rows, best_categories[:len(rows)]):  
        # make sure alignment between rows and best_categories is correct
        all_rows.append((lang, row["message"], cat))

# Group into category → language → messages
for lang, msg, cat in all_rows:
    category_to_lang_msgs[cat][lang].append(msg)

# Step 2: sort categories by number of total messages
sorted_categories = sorted(
    category_to_lang_msgs.items(),
    key=lambda x: sum(len(msgs) for msgs in x[1].values()),
    reverse=True
)

# Step 3: pretty print with language distributions + random samples
for cat, lang_dict in sorted_categories:
    total_msgs = sum(len(msgs) for msgs in lang_dict.values())
    print(f"\nCategory {cat} ({total_msgs} messages):")
    
    for lang, msgs in sorted(lang_dict.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  Language {lang}: {len(msgs)} messages")
        sample_msgs = random.sample(msgs, min(5, len(msgs)))
        for m in sample_msgs:
            print("    -", m.replace("\n", " ")[:120], "...")


Category Syntax Error (15169 messages):
  Language csharp: 6428 messages
    - Too many characters in character literal ...
    - ; expected ...
    - Invalid expression term '<' ...
    - ) expected ...
    - Invalid expression term '<' ...
  Language python: 3338 messages
    - invalid syntax (45103a3d1e8835f14eb515cc3d5b97ca_1_1200_original.py, line 1) ...
    - invalid syntax (2761546fafe579bf61c0c58400cc7efd_5_1200_varies-according-to-vulnerability.py, line 1) ...
    - invalid syntax (058f33787f8ac4b9cfabcb0e316c4eae_1_1200_from-security-standpoint.py, line 1) ...
    - invalid syntax (0f2aee9f1a95e1d6618f3249070da5d1_0_1200_from-security-standpoint.py, line 1) ...
    - invalid syntax (3d2d4d86f1d88df9f485f691ad26c9d8_1_1200_from-security-standpoint.py, line 1) ...
  Language c: 2855 messages
    - unknown type name 'sudo' ...
    - stray '\302' in program ...
    - expected identifier or '(' before 'switch' ...
    - 'front' undeclared (first use in this function) ...
    - ex

### Clustering (Not using this rn)

In [19]:
n_clusters = 5  

clustering = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)
labels = clustering.fit_predict(embeddings)

Exception ignored on calling ctypes callback function <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7f07d6ba6340>:
Traceback (most recent call last):
  File "/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
  File "/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
  File "/usr/lib64/python3.13/ctypes/__init__.py", line 361, in __init__
    self._handle = self._load_library(name, mode, handle, winmode)
  File "/usr/lib64/python3.13/ctypes/__init__.py", line 401, in _load_library
    return _dlopen(name, mode)
O

In [20]:
labels

array([3, 3, 3, ..., 2, 2, 2], shape=(24257,), dtype=int32)

In [21]:
clusters = defaultdict(list)
for label, msg in zip(labels, messages):
    clusters[label].append(msg)

In [22]:
for cid in range(n_clusters):
    print(f"\nCluster {cid} ({len(clusters[cid])} messages):")
    for m in clusters[cid][:5]:  # preview first 5 per cluster
        print("  -", m.replace("\n", " ")[:120], "...")


Cluster 0 (5286 messages):
  - invalid character '├' (U+251C) (f65c4c09f07f017ce0dcb8dfa9e8ac06_0_1200_act-as-security-specialist.py, line 2) ...
  - unexpected indent (68531143a127f874ab0bec63419932e8_1_1200_act-as-security-specialist.py, line 3) ...
  - invalid syntax (a07842b1a28cca9853c88c480edcbfd8_2_1200_act-as-security-specialist.py, line 1) ...
  - invalid syntax (6d2b944c19cb8924f509ed5a16cbfbbb_0_1200_act-as-security-specialist.py, line 1) ...
  - invalid syntax (6d2b944c19cb8924f509ed5a16cbfbbb_1_1200_act-as-security-specialist.py, line 1) ...

Cluster 1 (4317 messages):
  - ; expected ...
  - ) expected ...
  - ; expected ...
  - ; expected ...
  - } expected ...

Cluster 2 (9962 messages):
  - The modifier 'private' is not valid for this item ...
  - The modifier 'private' is not valid for this item ...
  - The modifier 'private' is not valid for this item ...
  - The modifier 'private' is not valid for this item ...
  - The modifier 'public' is not valid for this item ..

In [23]:
from collections import Counter, defaultdict

# Attach labels to rows (messages, symbols, types, etc.)
all_rows = []
i = 0
for key, value in linting_results.items():
    for row in value:
        row_copy = dict(row)  # keep original fields
        row_copy["language"] = key
        row_copy["cluster"] = int(labels[i])  # add cluster label
        all_rows.append(row_copy)
        i += 1

print(f"Total rows with clusters: {len(all_rows)}")

# Group by cluster
clusters = defaultdict(list)
for row in all_rows:
    clusters[row["cluster"]].append(row)

# --- Analysis per cluster ---
for cid, rows in clusters.items():
    print("\n" + "="*60)
    print(f"Cluster {cid} (size={len(rows)})")
    print("="*60)

    # Count messages
    msg_counts = Counter([r["message"] for r in rows if "message" in r])
    print("Top 5 messages:")
    for msg, count in msg_counts.most_common(5):
        print(f"{count:5d}  {msg}")

    # Count symbols
    sym_counts = Counter([r["symbol"] for r in rows if "symbol" in r])
    print("Top symbols:")
    for sym, count in sym_counts.most_common(3):
        print(f"{count:5d}  {sym}")

    # Count types
    type_counts = Counter([r["type"] for r in rows if "type" in r])
    print("Top types:")
    for typ, count in type_counts.most_common(3):
        print(f"{count:5d}  {typ}")

    # Breakdown by language
    lang_counts = Counter([r["language"] for r in rows])
    print("Languages in cluster:")
    for lang, count in lang_counts.most_common():
        print(f"{lang:12s} {count}")


Total rows with clusters: 24257

Cluster 3 (size=4058)
Top 5 messages:
  709  Invalid expression term '<'
  683  Syntax error, ',' expected
  366  stray '\302' in program
  357  Invalid expression term '='
  250  stray '\342' in program
Top symbols:
 3933  syntax-error
  125  Not valid php file
Top types:
 4058  syntax-error
Languages in cluster:
csharp       2867
c            1066
php          125

Cluster 2 (size=9962)
Top 5 messages:
 1241  Parsing error: Unexpected token <
  927  Identifier expected
  471  Type or namespace definition, or end-of-file expected
  466  Parsing error: Unexpected token install
  422  Too many characters in character literal
Top symbols:
 9962  syntax-error
Top types:
 9962  syntax-error
Languages in cluster:
javascript   4075
c            3480
csharp       2404
java         3

Cluster 1 (size=4317)
Top 5 messages:
 2679  ; expected
  984  } expected
  340  ) expected
  267  { expected
   47  Expected expression
Top symbols:
 4317  syntax-error
Top types